# Notebook 02 — Data Cleaning & ETL Pipeline
**Project:** Credit Risk & Loan Default Analysis  
**Objective:** Clean raw dataset and output analysis-ready file to data/processed/  
**Rule:** Every transformation step is logged below.

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

RAW_PATH = '../data/raw/lending_club_loans.csv'
PROCESSED_PATH = '../data/processed/loans_cleaned.csv'

df = pd.read_csv(RAW_PATH, low_memory=False)
print(f'Raw shape: {df.shape}')

Raw shape: (2260701, 151)


## Step 1 — Select Relevant Columns

In [9]:
KEEP_COLS = [
    'loan_amnt', 'funded_amnt', 'term', 'int_rate', 'installment',
    'grade', 'sub_grade', 'emp_length', 'home_ownership', 'annual_inc',
    'verification_status', 'loan_status', 'purpose', 'addr_state',
    'dti', 'delinq_2yrs', 'fico_range_low', 'fico_range_high',
    'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
    'issue_d', 'earliest_cr_line'
]

df = df[[c for c in KEEP_COLS if c in df.columns]].copy()
print(f'After column selection: {df.shape}')

After column selection: (2260701, 25)


## Step 2 — Drop Rows with No Loan Status (unusable records)

In [10]:
before = len(df)
df = df[df['loan_status'].notna()]
print(f'Dropped {before - len(df):,} rows with null loan_status')

# Keep only completed loans (exclude current/in-progress)
completed_statuses = [
    'Fully Paid', 'Charged Off', 'Default',
    'Does not meet the credit policy. Status:Fully Paid',
    'Does not meet the credit policy. Status:Charged Off'
]
before = len(df)
df = df[df['loan_status'].isin(completed_statuses)]
print(f'Dropped {before - len(df):,} in-progress/current loans')
print(f'Remaining: {len(df):,} completed loans')

Dropped 33 rows with null loan_status
Dropped 912,569 in-progress/current loans
Remaining: 1,348,099 completed loans


## Step 3 — Create Default Flag (Target Variable)

In [11]:
default_statuses = [
    'Charged Off', 'Default',
    'Does not meet the credit policy. Status:Charged Off'
]
df['default_flag'] = df['loan_status'].isin(default_statuses).astype(int)
print(f'Default rate: {df["default_flag"].mean():.2%}')
print(df['default_flag'].value_counts())

Default rate: 19.98%
default_flag
0    1078739
1     269360
Name: count, dtype: int64


## Step 4 — Fix Data Types

In [12]:
# int_rate: remove % and convert to float
if df['int_rate'].dtype == object:
    df['int_rate'] = df['int_rate'].str.replace('%', '').str.strip().astype(float)
    print('int_rate: converted to float')

# revol_util: remove % and convert to float
if df['revol_util'].dtype == object:
    df['revol_util'] = df['revol_util'].str.replace('%', '').str.strip()
    df['revol_util'] = pd.to_numeric(df['revol_util'], errors='coerce')
    print('revol_util: converted to float')

# term: extract numeric value
if df['term'].dtype == object:
    df['term'] = df['term'].str.strip().str.extract(r'(\d+)').astype(float)
    print('term: extracted numeric months')

# emp_length: extract numeric value
if df['emp_length'].dtype == object:
    df['emp_length'] = df['emp_length'].str.extract(r'(\d+)').astype(float)
    print('emp_length: extracted numeric years')

# Date columns
df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y', errors='coerce')
df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y', errors='coerce')
print('Dates: parsed to datetime')

Dates: parsed to datetime


## Step 5 — Handle Missing Values

In [13]:
missing_before = df.isnull().sum()
print('Missing values before treatment:')
print(missing_before[missing_before > 0])

Missing values before treatment:
emp_length          78550
annual_inc              4
dti                   374
delinq_2yrs            29
open_acc               29
pub_rec                29
revol_util            897
total_acc              29
earliest_cr_line       29
dtype: int64


In [14]:
# Numeric columns: force to float first, then fill with median
numeric_cols = ['annual_inc', 'dti', 'revol_util', 'emp_length', 'delinq_2yrs', 'open_acc', 'pub_rec', 'total_acc']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        median_val = df[col].median(skipna=True)
        filled = df[col].isna().sum()
        df[col] = df[col].fillna(median_val)
        if filled > 0:
            print(f'{col}: filled {filled:,} nulls with median ({median_val:.2f})')

# Categorical columns: fill with mode
cat_cols = ['home_ownership', 'verification_status', 'purpose']
for col in cat_cols:
    if col in df.columns:
        mode_val = df[col].mode()[0]
        filled = df[col].isna().sum()
        df[col] = df[col].fillna(mode_val)
        if filled > 0:
            print(f'{col}: filled {filled:,} nulls with mode ({mode_val})')

annual_inc: filled 4 nulls with median (65000.00)
dti: filled 374 nulls with median (17.61)
revol_util: filled 897 nulls with median (52.20)
emp_length: filled 1,348,099 nulls with median (nan)
delinq_2yrs: filled 29 nulls with median (0.00)
open_acc: filled 29 nulls with median (11.00)
pub_rec: filled 29 nulls with median (0.00)
total_acc: filled 29 nulls with median (23.00)


## Step 6 — Outlier Treatment

In [15]:
# Cap annual_inc at 99th percentile (extreme outliers skew analysis)
p99 = df['annual_inc'].quantile(0.99)
outliers = (df['annual_inc'] > p99).sum()
df['annual_inc'] = df['annual_inc'].clip(upper=p99)
print(f'annual_inc: capped {outliers:,} values above 99th percentile ({p99:,.0f})')

# Cap dti at 99th percentile
p99_dti = df['dti'].quantile(0.99)
outliers_dti = (df['dti'] > p99_dti).sum()
df['dti'] = df['dti'].clip(upper=p99_dti)
print(f'dti: capped {outliers_dti:,} values above 99th percentile ({p99_dti:.1f})')

annual_inc: capped 13,473 values above 99th percentile (251,000)
dti: capped 13,479 values above 99th percentile (38.5)


## Step 7 — Standardise Categorical Values

In [16]:
# Uppercase and strip whitespace
str_cols = ['grade', 'home_ownership', 'purpose', 'verification_status', 'addr_state']
for col in str_cols:
    if col in df.columns:
        df[col] = df[col].str.upper().str.strip()
        print(f'{col}: standardised to uppercase')

grade: standardised to uppercase
home_ownership: standardised to uppercase
purpose: standardised to uppercase
verification_status: standardised to uppercase
addr_state: standardised to uppercase


## Step 8 — Feature Engineering

In [17]:
# Average FICO score
df['fico_avg'] = (df['fico_range_low'] + df['fico_range_high']) / 2
print('Created: fico_avg')

# Credit age in years
df['credit_age_years'] = ((df['issue_d'] - df['earliest_cr_line']).dt.days / 365).round(1)
df['credit_age_years'] = df['credit_age_years'].clip(lower=0)
print('Created: credit_age_years')

# Income band
df['income_band'] = pd.cut(
    df['annual_inc'],
    bins=[0, 40000, 80000, 120000, float('inf')],
    labels=['Low', 'Medium', 'High', 'Very High']
)
print('Created: income_band')

# DTI band
df['dti_band'] = pd.cut(
    df['dti'],
    bins=[0, 10, 20, 30, float('inf')],
    labels=['Low', 'Moderate', 'High', 'Very High']
)
print('Created: dti_band')

# Issue year and month for time-series
df['issue_year'] = df['issue_d'].dt.year
df['issue_month'] = df['issue_d'].dt.month
print('Created: issue_year, issue_month')

Created: fico_avg
Created: credit_age_years
Created: income_band
Created: dti_band
Created: issue_year, issue_month


## Step 9 — Final Validation

In [18]:
print(f'Final shape: {df.shape}')
print(f'Default rate: {df["default_flag"].mean():.2%}')
print(f'Remaining nulls: {df.isnull().sum().sum()}')
print(f'\nColumns: {list(df.columns)}')
df.head()

Final shape: (1348099, 32)
Default rate: 19.98%
Remaining nulls: 1349401

Columns: ['loan_amnt', 'funded_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'loan_status', 'purpose', 'addr_state', 'dti', 'delinq_2yrs', 'fico_range_low', 'fico_range_high', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'issue_d', 'earliest_cr_line', 'default_flag', 'fico_avg', 'credit_age_years', 'income_band', 'dti_band', 'issue_year', 'issue_month']


,loan_amnt,funded_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,...,total_acc,issue_d,earliest_cr_line,default_flag,fico_avg,credit_age_years,income_band,dti_band,issue_year,issue_month
0,3600.0,3600.0,36 months,13.99,123.03,C,C4,NaN,MORTGAGE,55000.0,...,13.0,2015-12-01,2003-08-01,0,677.0,12.3,Medium,Low,2015,12
1,24700.0,24700.0,36 months,11.99,820.28,C,C1,NaN,MORTGAGE,65000.0,...,38.0,2015-12-01,1999-12-01,0,717.0,16.0,Medium,Moderate,2015,12
2,20000.0,20000.0,60 months,10.78,432.66,B,B4,NaN,MORTGAGE,63000.0,...,18.0,2015-12-01,2000-08-01,0,697.0,15.3,Medium,Moderate,2015,12
4,10400.0,10400.0,60 months,22.45,289.91,F,F1,NaN,MORTGAGE,104433.0,...,35.0,2015-12-01,1998-06-01,0,697.0,17.5,High,High,2015,12
5,11950.0,11950.0,36 months,13.44,405.18,C,C3,NaN,RENT,34000.0,...,6.0,2015-12-01,1987-10-01,0,692.0,28.2,Low,Moderate,2015,12


## Step 10 — Export to Processed

In [19]:
df.to_csv(PROCESSED_PATH, index=False)
print(f'Saved to {PROCESSED_PATH}')
print('ETL pipeline complete. Proceed to 03_eda.ipynb')

Saved to ../data/processed/loans_cleaned.csv
ETL pipeline complete. Proceed to 03_eda.ipynb
